In [ ]:
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import scipy.interpolate as spi
from scipy.io import savemat
import time

import desc
!pip show desc-opt

### output3D：三维平衡输出

本 notebook 读取 DESC 生成的三维平衡 `.hdf5` 文件，并输出 MATLAB 可读取的三维平衡数据。请使用 DESC 0.17.0。

最终将在目标文件夹中生成 `standard3D.mat` 和 `plot3D.mat`。

### 第一步：设置输入和输出路径

在 `1.1代码块` 中设置 `inputPath` 和 `outputPath`。

`inputPath` 是已经计算好的 DESC 平衡 `.hdf5` 文件路径；`outputPath` 是 `.mat` 文件的输出文件夹。运行后会自动创建输出文件夹。

In [ ]:
################## 1.1代码块 ##################

from pathlib import Path

inputPath = r'/mnt/c/Users/Desktop/EQ.hdf5'
outputPath = r'/mnt/c/Users/Desktop/EQ/'

folder_path = Path(outputPath)
folder_path.mkdir(exist_ok=True)

### 第二步：定义输出函数

运行 `2.1代码块` 定义后续输出所需函数。本步骤不会立即生成文件。

In [ ]:
################## 2.1代码块 ##################

def getRhoThetaPESTPhi(rho_num, rho_min, rho_max, theta_num, phi_num, fullTorus):
    NFP = eq.NFP
    rho_grid = np.linspace(rho_min, rho_max, rho_num)

    dtheta_pest = 2*np.pi/theta_num
    theta_pest_min = -np.pi + dtheta_pest/2
    theta_pest_max =  np.pi - dtheta_pest/2
    theta_pest = np.linspace(theta_pest_min, theta_pest_max, theta_num)

    if fullTorus == 0:
        dphi = 2*np.pi/NFP/phi_num
        phi_min = -np.pi/NFP + dphi/2
        phi_max =  np.pi/NFP - dphi/2
    if fullTorus == 1:
        dphi = 2*np.pi/phi_num
        phi_min = -np.pi + dphi/2
        phi_max =  np.pi - dphi/2

    phi_grid = np.linspace(phi_min, phi_max, phi_num)

    return rho_grid, theta_pest, phi_grid


def makeNodes(rho_values, theta_values, phi_values):
    rr, tt, pp = np.meshgrid(rho_values, theta_values, phi_values, indexing='ij')
    return np.stack([rr.ravel(), tt.ravel(), pp.ravel()], axis=1)


def makeGrid(nodes, coordinates, NFP, fullTorus):
    if fullTorus == 0:
        phi_period = 2*np.pi/NFP
    if fullTorus == 1:
        phi_period = 2*np.pi

    return desc.grid.Grid(
        nodes,
        coordinates=coordinates,
        period=(np.inf, 2*np.pi, phi_period),
        NFP=NFP,
    )


def uniqueList(values):
    out = []
    seen = set()
    for value in values:
        if value not in seen:
            out.append(value)
            seen.add(value)
    return out


def dotVec(a, b):
    return np.sum(a * b, axis=-1)


def scalarField(field, name, shape):
    return np.asarray(field[name]).reshape(shape)


def vectorField(field, name, shape):
    return np.asarray(field[name]).reshape(shape + (3,))


def linearCombination(coeffs, bases, shape):
    out = np.zeros(shape + (3,))
    for coeff, basis in zip(coeffs, bases):
        out += coeff[..., None] * basis
    return out


def derivativeCombination(weights, derivs, shape):
    out = np.zeros(shape + (3,))
    for weight, deriv in zip(weights, derivs):
        out += weight[..., None] * deriv
    return out


def addDescMetricValues(field, values, shape):
    desc_con = {
        'rr': 'g^rr', 'rt': 'g^rt', 'rz': 'g^rz',
        'tt': 'g^tt', 'tz': 'g^tz', 'zz': 'g^zz',
    }
    for comp, var in desc_con.items():
        values[f'gcon_{comp}'] = scalarField(field, var, shape)
        for coord in ['r', 't', 'z']:
            values[f'dgcon_{comp}_d{coord}'] = scalarField(field, f'{var}_{coord}', shape)

    desc_cov = {
        'rr': 'g_rr', 'rt': 'g_rt', 'rz': 'g_rz',
        'tt': 'g_tt', 'tz': 'g_tz', 'zz': 'g_zz',
    }
    for comp, var in desc_cov.items():
        values[f'gcov_{comp}'] = scalarField(field, var, shape)

    e = {
        'r': vectorField(field, 'e_rho', shape),
        't': vectorField(field, 'e_theta', shape),
        'z': vectorField(field, 'e_zeta', shape),
    }
    de = {
        'r': {
            'r': vectorField(field, 'e_rho_r', shape),
            't': vectorField(field, 'e_rho_t', shape),
            'z': vectorField(field, 'e_rho_z', shape),
        },
        't': {
            'r': vectorField(field, 'e_rho_t', shape),
            't': vectorField(field, 'e_theta_t', shape),
            'z': vectorField(field, 'e_theta_z', shape),
        },
        'z': {
            'r': vectorField(field, 'e_rho_z', shape),
            't': vectorField(field, 'e_theta_z', shape),
            'z': vectorField(field, 'e_zeta_z', shape),
        },
    }
    comps = [('rr', 'r', 'r'), ('rt', 'r', 't'), ('rz', 'r', 'z'),
             ('tt', 't', 't'), ('tz', 't', 'z'), ('zz', 'z', 'z')]
    for comp, i, j in comps:
        for coord in ['r', 't', 'z']:
            values[f'dgcov_{comp}_d{coord}'] = dotVec(de[coord][i], e[j]) + dotVec(e[i], de[coord][j])


def pestTransformCoefficients(field, shape):
    lr = scalarField(field, 'lambda_r', shape)
    lt = scalarField(field, 'lambda_t', shape)
    lz = scalarField(field, 'lambda_z', shape)
    lrr = scalarField(field, 'lambda_rr', shape)
    lrt = scalarField(field, 'lambda_rt', shape)
    lrz = scalarField(field, 'lambda_rz', shape)
    ltt = scalarField(field, 'lambda_tt', shape)
    ltz = scalarField(field, 'lambda_tz', shape)
    lzz = scalarField(field, 'lambda_zz', shape)

    one = np.ones(shape)
    zero = np.zeros(shape)
    D = one + lt

    a = -lr / D
    b = one / D
    c = -lz / D

    T = {
        'x': [one, a, zero],
        'y': [zero, b, zero],
        'z': [zero, c, one],
    }

    lr_m = {'r': lrr, 't': lrt, 'z': lrz}
    lt_m = {'r': lrt, 't': ltt, 'z': ltz}
    lz_m = {'r': lrz, 't': ltz, 'z': lzz}

    da = {m: -(lr_m[m] * D - lr * lt_m[m]) / D**2 for m in ['r', 't', 'z']}
    db = {m: -lt_m[m] / D**2 for m in ['r', 't', 'z']}
    dc = {m: -(lz_m[m] * D - lz * lt_m[m]) / D**2 for m in ['r', 't', 'z']}

    cov_C = {
        'x': [one, a, zero],
        'y': [zero, b, zero],
        'z': [zero, c, one],
    }
    cov_dC_old = {
        m: {
            'x': [zero, da[m], zero],
            'y': [zero, db[m], zero],
            'z': [zero, dc[m], zero],
        } for m in ['r', 't', 'z']
    }

    con_L = {
        'x': [one, zero, zero],
        'y': [lr, D, lz],
        'z': [zero, zero, one],
    }
    con_dL_old = {
        m: {
            'x': [zero, zero, zero],
            'y': [lr_m[m], lt_m[m], lz_m[m]],
            'z': [zero, zero, zero],
        } for m in ['r', 't', 'z']
    }

    return T, cov_C, cov_dC_old, con_L, con_dL_old


def buildNewBasis(basis_old, dbasis_old, T, coeffs, dcoeffs_old, shape):
    old_coords = ['r', 't', 'z']
    new_coords = ['x', 'y', 'z']

    basis = {}
    deriv = {coord: {} for coord in new_coords}

    for name in new_coords:
        basis[name] = linearCombination(coeffs[name], basis_old, shape)

    for deriv_coord in new_coords:
        for basis_name in new_coords:
            dcoeff = []
            for old_i in range(3):
                temp = np.zeros(shape)
                for weight, old_coord in zip(T[deriv_coord], old_coords):
                    temp += weight * dcoeffs_old[old_coord][basis_name][old_i]
                dcoeff.append(temp)

            out = linearCombination(dcoeff, basis_old, shape)
            for coeff, old_i in zip(coeffs[basis_name], range(3)):
                old_derivs = [dbasis_old[old_coord][old_i] for old_coord in old_coords]
                out += coeff[..., None] * derivativeCombination(T[deriv_coord], old_derivs, shape)
            deriv[deriv_coord][basis_name] = out

    return basis, deriv


def addPestMetricValues(field, values, shape, prefix=''):
    T, cov_C, cov_dC_old, con_L, con_dL_old = pestTransformCoefficients(field, shape)

    pest_cov_direct = {
        'xx': 'g_rr|PEST', 'xy': 'g_rv|PEST', 'xz': 'g_rp|PEST',
        'yy': 'g_vv|PEST', 'yz': 'g_vp|PEST', 'zz': 'g_pp|PEST',
    }
    for comp, var in pest_cov_direct.items():
        values[f'{prefix}PESTcov_{comp}'] = scalarField(field, var, shape)

    cov_old = [
        vectorField(field, 'e_rho', shape),
        vectorField(field, 'e_theta', shape),
        vectorField(field, 'e_zeta', shape),
    ]
    dcov_old = {
        'r': [
            vectorField(field, 'e_rho_r', shape),
            vectorField(field, 'e_rho_t', shape),
            vectorField(field, 'e_rho_z', shape),
        ],
        't': [
            vectorField(field, 'e_rho_t', shape),
            vectorField(field, 'e_theta_t', shape),
            vectorField(field, 'e_theta_z', shape),
        ],
        'z': [
            vectorField(field, 'e_rho_z', shape),
            vectorField(field, 'e_theta_z', shape),
            vectorField(field, 'e_zeta_z', shape),
        ],
    }
    pest_cov_basis, pest_cov_deriv = buildNewBasis(cov_old, dcov_old, T, cov_C, cov_dC_old, shape)

    comps = [('xx', 'x', 'x'), ('xy', 'x', 'y'), ('xz', 'x', 'z'),
             ('yy', 'y', 'y'), ('yz', 'y', 'z'), ('zz', 'z', 'z')]
    for comp, i, j in comps:
        for deriv_coord in ['x', 'y', 'z']:
            suffix = f'p{deriv_coord}'
            values[f'{prefix}dPESTcov_{comp}_{suffix}'] = (
                dotVec(pest_cov_deriv[deriv_coord][i], pest_cov_basis[j])
                + dotVec(pest_cov_basis[i], pest_cov_deriv[deriv_coord][j])
            )

    pest_cov_direct_deriv = {
        ('xx', 'py'): '(g_rr_v)|PEST',
        ('xx', 'pz'): '(g_rr_p)|PEST',
        ('xy', 'pz'): '(g_rv_p)|PEST',
        ('yy', 'px'): '(g_vv_r)|PEST',
        ('yy', 'pz'): '(g_vv_p)|PEST',
        ('zz', 'py'): '(g_pp_v)|PEST',
    }
    for (comp, suffix), var in pest_cov_direct_deriv.items():
        values[f'{prefix}dPESTcov_{comp}_{suffix}'] = scalarField(field, var, shape)

    pest_con_direct = {
        'xx': 'g^rr', 'xy': 'g^rv', 'xz': 'g^rz', 'zz': 'g^zz',
    }
    for comp, var in pest_con_direct.items():
        values[f'{prefix}PESTcon_{comp}'] = scalarField(field, var, shape)

    con_old = [
        vectorField(field, 'e^rho', shape),
        vectorField(field, 'e^theta', shape),
        vectorField(field, 'e^zeta', shape),
    ]
    dcon_old = {
        'r': [
            vectorField(field, 'e^rho_r', shape),
            vectorField(field, 'e^theta_r', shape),
            vectorField(field, 'e^zeta_r', shape),
        ],
        't': [
            vectorField(field, 'e^rho_t', shape),
            vectorField(field, 'e^theta_t', shape),
            vectorField(field, 'e^zeta_t', shape),
        ],
        'z': [
            vectorField(field, 'e^rho_z', shape),
            vectorField(field, 'e^theta_z', shape),
            vectorField(field, 'e^zeta_z', shape),
        ],
    }
    pest_con_basis, pest_con_deriv = buildNewBasis(con_old, dcon_old, T, con_L, con_dL_old, shape)

    values[f'{prefix}PESTcon_yy'] = dotVec(pest_con_basis['y'], pest_con_basis['y'])
    values[f'{prefix}PESTcon_yz'] = dotVec(pest_con_basis['y'], pest_con_basis['z'])

    for comp, i, j in comps:
        for deriv_coord in ['x', 'y', 'z']:
            suffix = f'p{deriv_coord}'
            values[f'{prefix}dPESTcon_{comp}_{suffix}'] = (
                dotVec(pest_con_deriv[deriv_coord][i], pest_con_basis[j])
                + dotVec(pest_con_basis[i], pest_con_deriv[deriv_coord][j])
            )

    pest_con_direct_deriv = {
        ('xx', 'py'): '(g^rr_v)|PEST',
        ('xx', 'pz'): '(g^rr_p)|PEST',
        ('xy', 'py'): '(g^rv_v)|PEST',
        ('xy', 'pz'): '(g^rv_p)|PEST',
        ('xz', 'py'): '(g^rz_v)|PEST',
        ('xz', 'pz'): '(g^rz_p)|PEST',
    }
    for (comp, suffix), var in pest_con_direct_deriv.items():
        values[f'{prefix}dPESTcon_{comp}_{suffix}'] = scalarField(field, var, shape)


def pestMetricComputeList():
    direct = [
        'lambda_r', 'lambda_t', 'lambda_z', 'lambda_rr', 'lambda_rt', 'lambda_rz', 'lambda_tt', 'lambda_tz', 'lambda_zz',
        'e_rho', 'e_theta', 'e_zeta', 'e_theta_PEST', 'e_phi|r,v', 'e_rho_r', 'e_rho_t', 'e_rho_z', 'e_theta_t', 'e_theta_z', 'e_zeta_z',
        'e^rho', 'e^theta', 'e^zeta',
        'e^rho_r', 'e^rho_t', 'e^rho_z',
        'e^theta_r', 'e^theta_t', 'e^theta_z',
        'e^zeta_r', 'e^zeta_t', 'e^zeta_z',
        'g_rr|PEST', 'g_rv|PEST', 'g_rp|PEST', 'g_vv|PEST', 'g_vp|PEST', 'g_pp|PEST',
        'g^rr', 'g^rv', 'g^rz', 'g^zz',
    ]
    direct.extend([
        '(g_rr_v)|PEST', '(g_rr_p)|PEST', '(g_rv_p)|PEST', '(g_vv_r)|PEST', '(g_vv_p)|PEST', '(g_pp_v)|PEST',
        '(g^rr_v)|PEST', '(g^rr_p)|PEST', '(g^rv_v)|PEST', '(g^rv_p)|PEST', '(g^rz_v)|PEST', '(g^rz_p)|PEST',
    ])
    return uniqueList(direct)


def standardComputeList():
    direct = [
        'R', 'Z',
        'sqrt(g)', 'sqrt(g)_PEST', '(sqrt(g)_PEST_r)|PEST', '(sqrt(g)_PEST_v)|PEST', '(sqrt(g)_PEST_p)|PEST',
        'J_parallel',
        '|B|', '|B|_r', '|B|_t', '|B|_z', '|B|_rr', '|B|_rt', '|B|_rz', '|B|_tt', '|B|_tz', '|B|_zz',
        'lambda', 'lambda_r', 'lambda_t', 'lambda_z', 'lambda_rr', 'lambda_rt', 'lambda_rz', 'lambda_tt', 'lambda_tz', 'lambda_zz',
        'rho', 'q', 'iota', 'iota_r', 'iota_rr', 'theta',
        'e_rho', 'e_theta', 'e_zeta', 'e_rho_r', 'e_rho_t', 'e_rho_z', 'e_theta_t', 'e_theta_z', 'e_zeta_z',
    ]
    for var in ['g^rr', 'g^rt', 'g^rz', 'g^tt', 'g^tz', 'g^zz']:
        direct.append(var)
        direct.extend([f'{var}_r', f'{var}_t', f'{var}_z'])
    direct.extend(['g_rr', 'g_rt', 'g_rz', 'g_tt', 'g_tz', 'g_zz'])
    direct += pestMetricComputeList()
    return uniqueList(direct)


def collectStandardValues(field, shape):
    values = {}

    values['R'] = scalarField(field, 'R', shape)
    values['Z'] = scalarField(field, 'Z', shape)
    values['JDESC'] = scalarField(field, 'sqrt(g)', shape)
    values['JPEST'] = scalarField(field, 'sqrt(g)_PEST', shape)
    values['dJPEST_drho'] = scalarField(field, '(sqrt(g)_PEST_r)|PEST', shape)
    values['dJPEST_dtheta'] = scalarField(field, '(sqrt(g)_PEST_v)|PEST', shape)
    values['dJPEST_dphi'] = scalarField(field, '(sqrt(g)_PEST_p)|PEST', shape)
    values['J_parallel'] = scalarField(field, 'J_parallel', shape)

    b_map = {
        'B': '|B|', 'dB_dr': '|B|_r', 'dB_dt': '|B|_t', 'dB_dz': '|B|_z',
        'dB_dr2': '|B|_rr', 'dB_drdt': '|B|_rt', 'dB_drdz': '|B|_rz',
        'dB_dt2': '|B|_tt', 'dB_dtdz': '|B|_tz', 'dB_dz2': '|B|_zz',
    }
    for out_name, desc_name in b_map.items():
        values[out_name] = scalarField(field, desc_name, shape)

    lambda_map = {
        'lambda': 'lambda', 'dlambda_dr': 'lambda_r', 'dlambda_dt': 'lambda_t', 'dlambda_dz': 'lambda_z',
        'dlambda_dr2': 'lambda_rr', 'dlambda_drdt': 'lambda_rt', 'dlambda_drdz': 'lambda_rz',
        'dlambda_dt2': 'lambda_tt', 'dlambda_dtdz': 'lambda_tz', 'dlambda_dz2': 'lambda_zz',
    }
    for out_name, desc_name in lambda_map.items():
        values[out_name] = scalarField(field, desc_name, shape)

    addDescMetricValues(field, values, shape)
    addPestMetricValues(field, values, shape)

    e_rho = vectorField(field, 'e_rho', shape)
    e_theta = vectorField(field, 'e_theta', shape)
    lambda_r = scalarField(field, 'lambda_r', shape)
    lambda_t = scalarField(field, 'lambda_t', shape)
    e_rho_pest = e_rho - (lambda_r / (1.0 + lambda_t))[..., None] * e_theta
    values['dR_drho'] = e_rho_pest[..., 0]
    values['dZ_drho'] = e_rho_pest[..., 2]

    e_theta_pest = vectorField(field, 'e_theta_PEST', shape)
    e_phi_pest = vectorField(field, 'e_phi|r,v', shape)
    values['dR_dtheta'] = e_theta_pest[..., 0]
    values['dZ_dtheta'] = e_theta_pest[..., 2]
    values['dR_dphi'] = e_phi_pest[..., 0]
    values['dZ_dphi'] = e_phi_pest[..., 2]

    for name in ['rho', 'q', 'iota', 'iota_r', 'iota_rr']:
        values[name] = scalarField(field, name, shape)

    return values


def outputStandard3D(rho_grid, theta_pest, phi_grid, fullTorus, n_segments):
    NFP = eq.NFP
    rho_num = rho_grid.size
    theta_num = theta_pest.size
    phi_num = phi_grid.size
    full_shape = (rho_num, theta_num, phi_num)

    outputs = None
    theta_desc = np.zeros(full_shape)
    compute_list = standardComputeList()
    
    segment_size = rho_num // n_segments

    for seg in range(n_segments):
        
        print(seg)
        
        start = seg * segment_size
        end = (seg + 1) * segment_size
        rho_seg = rho_grid[start:end]
        seg_shape = (segment_size, theta_num, phi_num)

        nodes = makeNodes(rho_seg, theta_pest, phi_grid)
        field = eq.compute(compute_list, makeGrid(nodes, 'rvp', NFP, fullTorus))
        values = collectStandardValues(field, seg_shape)

        if outputs is None:
            outputs = {name: np.zeros(full_shape) for name in values}

        for name, value in values.items():
            outputs[name][start:end, :, :] = value
        theta_desc[start:end, :, :] = scalarField(field, 'theta', seg_shape)

    rho_mesh, theta_mesh, phi_mesh = np.meshgrid(rho_grid, theta_pest, phi_grid, indexing='ij')

    B00 = eq.compute('|B|', makeGrid(np.array([[0, 0, 0]]), 'rtz', NFP, fullTorus))['|B|']
    L00 = eq.compute('R', makeGrid(np.array([[0, 0, 0]]), 'rtz', NFP, fullTorus))['R']
    psit_max = eq.compute('psi', makeGrid(np.array([[1, 0, 0]]), 'rtz', NFP, fullTorus))['psi']

    # iotaOrders = eq.iota.basis.modes[:, 0]
    # iotaCoeffs = eq.iota.params

    # psipOrders = iotaOrders + 2
    # psipCoeffs = iotaCoeffs / psipOrders

    # dpsipdrOrders = iotaOrders + 1
    # dpsipdrCoeffs = iotaCoeffs

    # psipCoeffs = psipCoeffs * 2 * psit_max
    # dpsipdrCoeffs = dpsipdrCoeffs * 2 * psit_max

    # psipPoly = desc.profiles.PowerSeriesProfile(params=psipCoeffs, modes=psipOrders)
    # dpsipdrPoly = desc.profiles.PowerSeriesProfile(params=dpsipdrCoeffs, modes=dpsipdrOrders)

    # psipIndex = np.zeros((rho_grid.size, 3))
    # psipIndex[:, 0] = rho_grid

    # psip = np.array(psipPoly.compute(desc.grid.Grid(psipIndex)))
    # dpsipdr = np.array(dpsipdrPoly.compute(desc.grid.Grid(psipIndex)))

    # psip = np.repeat(psip[:, np.newaxis], theta_num, axis=1)
    # psip = np.repeat(psip[:, :, np.newaxis], phi_num, axis=2)
    # dpsipdr = np.repeat(dpsipdr[:, np.newaxis], theta_num, axis=1)
    # dpsipdr = np.repeat(dpsipdr[:, :, np.newaxis], phi_num, axis=2)

    if fullTorus == 0:
        phi_period = 2*np.pi/NFP
    if fullTorus == 1:
        phi_period = 2*np.pi

    save_dict = dict(outputs)
    save_dict['diota_dr'] = outputs['iota_r']
    save_dict['diota_dr2'] = outputs['iota_rr']

    save_dict.update({
        'theta_pest': theta_mesh,
        'theta_desc': theta_desc,
        'phi': phi_mesh,
        'NFP': NFP,
        'fullTorus': fullTorus,
        'phiPeriod': phi_period,
        'B00': B00,
        'L00': L00,
        'psit_max': psit_max,
        # 'psip': psip,
        # 'dpsip_dr': dpsipdr,
    })

    savemat(str(Path(outputPath) / 'standard3D.mat'), save_dict)



def outputPlot3D(rho_grid, theta_pest, phi_grid, fullTorus, n_segments):
    NFP = eq.NFP
    rho_num = rho_grid.size
    theta_num = theta_pest.size
    phi_num = phi_grid.size

    phi_stride = phi_num // 16
    phi_compute = phi_grid[::phi_stride]
    phi_compute_num = phi_compute.size
    
    matrix_list = ['q', 'R', 'Z', 'rho']
    
    matrix_num = len(matrix_list)
    output = np.zeros((matrix_num, rho_num, theta_num, phi_num))
    
    segment_size = int(rho_num / n_segments)
    
    for index in range(n_segments):
        
        print(index)
        
        start = index * segment_size
        end = (index + 1) * segment_size
        
        rho_segment = rho_grid[start:end]
        compute_shape = (segment_size, theta_num, phi_compute_num)
        
        nodes = makeNodes(rho_segment, theta_pest, phi_compute)
        field = eq.compute(matrix_list, makeGrid(nodes, 'rvp', NFP, fullTorus))
        
        for k in range(matrix_num):
            value = scalarField(field, matrix_list[k], compute_shape)
            output[k, start:end, :, :] = np.repeat(value, phi_stride, axis=2)
    
    path = outputPath + '/plot3D.mat'
    savemat(path,
            {'qplot': output[0],
             'Rplot': output[1],
             'Zplot': output[2],
             'rhoplot': output[3]})




### 第三步：读取和检查平衡

运行 `3.1代码块` 读取 `inputPath` 指定的平衡文件，并查看安全因子剖面、压强剖面、磁面形状和磁轴物理量。

In [ ]:
################## 3.1代码块 ##################

eq = desc.io.load(inputPath)

profileGrid = desc.grid.LinearGrid(L=200)
profileData = eq.compute(['q', 'p'], profileGrid)

plt.figure()
plt.plot(profileData['q'])
plt.title('Safety factor q')
plt.xlabel('rho')
plt.ylabel('q')

plt.figure()
plt.plot(profileData['p'])
plt.title('Pressure')
plt.xlabel('rho')
plt.ylabel('p')

desc.plotting.plot_surfaces(eq, 10, 36)
desc.plotting.plot_section(eq, '|F|_normalized', log=True)

u0 = 4 * np.pi * 1.0e-7
axisGrid = desc.grid.Grid(np.array([0, 0, 0]))
axisData = eq.compute(['|B|', 'p', 'R', 'Z'], axisGrid)

B0 = axisData['|B|'][0]
p0 = axisData['p'][0]
R0 = axisData['R'][0]
Z0 = axisData['Z'][0]
beta0 = p0 / (B0 * B0 / 2 / u0)

print('B0    =', B0)
print('R0    =', R0)
print('Z0    =', Z0)
print('beta0 =', beta0)

### 第四步：输出平衡

本步骤将平衡输出为两个 MATLAB 文件：`standard3D.mat` 和 `plot3D.mat`。

先运行 `4.1代码块` 设置标准输出网格，再运行 `4.2代码块` 生成 `rho`、`theta_PEST` 和 `phi` 网格。随后运行 `4.3代码块` 输出平衡文件。

运行 `4.4代码块` 得到高分辨率画图网格。

`4.1代码块` 设置标准输出网格：

1. `rhoMin`：输出区域的最小归一化径向坐标。
2. `rhoMax`：输出区域的最大归一化径向坐标。
3. `gridRho`：标准输出的径向网格数。
4. `gridTheta`：标准输出的 `theta_PEST` 网格数。
5. `gridPhi`：标准输出的 `phi` 网格数。
6. `fullTorus`：环向输出范围选项；`0` 表示一个场周期，`1` 表示全环。

In [ ]:
################## 4.1代码块 ##################

rhoMin = 0.08
rhoMax = 0.90
gridRho = 256
gridTheta = 32
gridPhi = 16
fullTorus = 0

`4.2代码块` 根据 `4.1代码块` 的网格设置，生成 `rho`、`theta_PEST` 和 `phi` 网格。

In [ ]:
################## 4.2代码块 ##################

rhoGrid, thetaPEST, phiGrid = getRhoThetaPESTPhi(gridRho, rhoMin, rhoMax, gridTheta, gridPhi, fullTorus)

`4.3代码块` 输出 `standard3D.mat`。这是三维平衡文件，包含磁场、几何、Jacobian、DESC/PEST 度规及其导数、剖面量和坐标网格。

`standardSegments` 是径向分段数，用于降低单次 `eq.compute` 的内存占用。`gridRho` 应能被 `standardSegments` 整除。

In [ ]:
################## 4.3代码块 ##################

start = time.perf_counter()

standardSegments = 8
outputStandard3D(rhoGrid, thetaPEST, phiGrid, fullTorus, standardSegments)

end = time.perf_counter()
elapsed_ms = (end - start) * 1000
print(f"耗时：{elapsed_ms:.3f} ms")

`4.4代码块` 输出 `plot3D.mat`，用于高分辨率绘图。

1. `plotRhoNum`：画图网格的径向网格数。
2. `plotThetaNum`：画图网格的极向网格数。
3. `plotSegments`：径向分段数，用于降低内存占用；`plotRhoNum` 应能被 `plotSegments` 整除。

In [ ]:
################## 4.4代码块 ##################

plotRhoNum = 1024
plotThetaNum = 1024
plotSegments = 8

start = time.perf_counter()

rhoGrid, thetaPEST, _ = getRhoThetaPESTPhi(plotRhoNum, rhoMin, rhoMax, plotThetaNum, phiGrid.size, fullTorus)
outputPlot3D(rhoGrid, thetaPEST, phiGrid, fullTorus, plotSegments)

end = time.perf_counter()
elapsed_ms = (end - start) * 1000
print(f"耗时：{elapsed_ms:.3f} ms")